# MA3001B • Semana 06: Dinámica Temporal y Análisis Espectral
## ACF, PACF, Correlación Cruzada (CCF) y Transformada de Fourier
**Profesor B: Modelos y TDA • Tecnológico de Monterrey**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alxcn/Models_MA3001B/blob/main/notebooks/N3_ACF_PACF_CCF_Espectral.ipynb)

---
### Objetivos de la Sesión
1. Calcular e interpretar la **ACF** y **PACF** para identificar estructuras autorregresivas (AR), de medias móviles (MA) y estacionales.
2. Comprender la **Correlación Cruzada (CCF)**, su convención de signos y el peligro de correlaciones espurias.
3. Extraer las periodicidades dominantes mediante el **periodograma de Fourier** y el método de Welch.
4. Contrastar los pesos de auto-atención del Transformer de la Semana 4 con los coeficientes de la PACF.


## 1. Problema Detonador: Correlación Espuria y Ventanas Arbitrarias
Dos caminatas aleatorias no estacionarias e independientes exhiben una correlación espuria cercana a 0.85. Además, la ventana $L=24$ se eligió por intuición. ¿Cómo descubrir rigurosamente los rezagos y ciclos verdaderos?

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import ccf, acf, pacf
from statsmodels.tsa.seasonal import STL
from scipy.signal import periodogram, find_peaks
import matplotlib.pyplot as plt

np.random.seed(42)

# Demostración de correlación espuria con caminatas aleatorias independientes:
n_sim = 200
rw1 = np.cumsum(np.random.randn(n_sim))
rw2 = np.cumsum(np.random.randn(n_sim))

corr_espuria = np.corrcoef(rw1, rw2)[0, 1]
print(f"Correlación de Pearson entre caminatas independientes no estacionarias: {corr_espuria:.3f}")
print("-> ¡Alerta! Parece haber alta correlación, pero es un artefacto de la tendencia común.")


## 2. Marco Teórico Mínimo

### 2.1 ACF y PACF
$$\rho(k) = \frac{\gamma(k)}{\gamma(0)}, \qquad \text{Bandas de Bartlett: } \pm \frac{1.96}{\sqrt{n}}$$

### 2.2 Periodograma y Frecuencia de Fourier
$$I(f_j) = \frac{1}{N} \left| \sum_{t=1}^N y_t e^{-2\pi i f_j t} \right|^2, \qquad T = \frac{1}{f_j}$$


### Configuración de Datos: Práctica (CO₂) o Serie del Reto
En la siguiente celda se carga la serie de práctica por defecto (**CO₂ mensual de Mauna Loa** de `statsmodels`).
Para aplicar el análisis a la serie de su equipo en el Reto, cambie la bandera `USAR_SERIE_RETO = True` y proporcione la ruta a su archivo CSV.

In [ ]:
# ==============================================================================
# CELDA SERIE_RETO: Alternar entre datos de práctica y datos del Reto
# ==============================================================================
USAR_SERIE_RETO = False  # Cambiar a True para usar la serie de su Reto

if USAR_SERIE_RETO:
    # Sustituya con la carga de la serie de su equipo:
    # df_reto = pd.read_csv("datos_reto.csv", index_col=0, parse_dates=True)
    # y = df_reto["columna_objetivo"].asfreq("MS").interpolate()
    print("Modo Reto activado.")
else:
    # Serie de práctica por defecto: CO2 de Mauna Loa (mensual interpolado)
    y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
    print(f"Cargada serie de práctica CO2: {len(y)} observaciones ({y.index.min().strftime('%Y-%m')} a {y.index.max().strftime('%Y-%m')})")


## 3. Ejemplo Aplicado Paso a Paso

### Paso 1: Correlogramas ACF y PACF sobre la Serie Diferenciada

In [ ]:
# 1) Serie diferenciada (estacionaria en media)
dy = y.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(dy, lags=36, alpha=0.05, ax=axes[0], title="ACF de la 1ª Diferencia")
plot_pacf(dy, lags=36, method="ywm", alpha=0.05, ax=axes[1], title="PACF de la 1ª Diferencia")

axes[0].grid(True, linestyle=":", alpha=0.6)
axes[1].grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# Picos en rezagos 12, 24 confirman la estacionalidad anual marcada


### Paso 2: Análisis Espectral con Periodograma

In [ ]:
# Remover tendencia para analizar componentes oscilatorias
stl_res = STL(y, period=12, robust=True).fit()
sin_tendencia = y - stl_res.trend

# Periodograma con fs=1.0 (1 muestra por mes)
freqs, potencias = periodogram(sin_tendencia, fs=1.0, window="hann", detrend="constant")
picos, _ = find_peaks(potencias, prominence=potencias.max() * 0.05)

periodos = np.round(1.0 / freqs[picos], 2)
print("=== PERIODOS DOMINANTES DETECTADOS (Meses) ===")
for f_val, t_val in zip(freqs[picos], periodos):
    print(f"Frecuencia f = {f_val:.4f} ciclos/mes -> Periodo T = {t_val:.1f} meses")

plt.figure(figsize=(9, 4))
plt.plot(freqs, potencias, color="#2AA198", lw=1.8, label="Densidad Espectral")
plt.plot(freqs[picos], potencias[picos], "x", color="#DC322F", markersize=8, label="Picos Dominantes")
plt.xlabel("Frecuencia (ciclos por mes)")
plt.ylabel("Densidad de Potencia")
plt.title("Periodograma de Fourier (CO2 sin tendencia)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


### Paso 3: Correlación Cruzada (CCF) y Convención de Signos
Verifiquemos qué argumento lidera en `ccf(y, x)`: si $x$ adelanta a $y$ por 3 periodos ($y_t \approx x_{t-3}$), el pico debe estar en el rezago positivo $k=3$ al pasar `ccf(y, x)`.

In [ ]:
# Simulación controlada para confirmar la convención de signo de statsmodels:
rng = np.random.default_rng(42)
x_lider = rng.normal(size=500)
# y_rezagado va 3 pasos detrás de x_lider:
y_rezagado = np.roll(x_lider, 3) + 0.1 * rng.normal(size=500)

ccf_yx = ccf(y_rezagado, x_lider, adjusted=False)[:10]
ccf_xy = ccf(x_lider, y_rezagado, adjusted=False)[:10]

print(f"Argmax en ccf(y_rezagado, x_lider): {np.argmax(ccf_yx)} (¡Debe ser 3!)")
print(f"Argmax en ccf(x_lider, y_rezagado): {np.argmax(ccf_xy)}")

# Aplicación a datos macroeconómicos reales: PIB vs Desempleo
macro = sm.datasets.macrodata.load_pandas().data
dgdp = np.log(macro["realgdp"]).diff().dropna()
dunemp = macro["unemp"].diff().dropna()

ccf_macro = ccf(dunemp, dgdp, adjusted=False)[:8]
print(f"CCF Cambio Desempleo vs Crecimiento PIB (rezagos 0 a 5): {np.round(ccf_macro[:6], 3)}")


### Paso 4: Contraste Atención del Transformer vs. PACF

In [ ]:
# Calculamos la PACF absoluta sobre la serie diferenciada hasta 24 rezagos
pacf_vals = np.abs(pacf(dy, nlags=24, method="ywm")[1:])  # Rezagos 1 a 24
rezagos = np.arange(1, 25)

plt.figure(figsize=(9, 4))
plt.bar(rezagos - 0.2, pacf_vals, width=0.4, label="|PACF| (Dependencia Lineal Directa)", color="#268BD2", alpha=0.85)
# Simulamos los pesos de atención normalizados
attn_sim = np.exp(-rezagos / 12) + 0.5 * (rezagos == 12)
attn_sim = attn_sim / attn_sim.sum()
plt.bar(rezagos + 0.2, attn_sim * 3, width=0.4, label="Atención Transformer (Escalada)", color="#CB4B16", alpha=0.85)

plt.xlabel("Rezago (k)")
plt.ylabel("Magnitud")
plt.title("Comparación: |PACF| vs. Perfil de Atención del Transformer")
plt.xticks(rezagos[::2])
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


## 4. Actividad del Reto y Entregable
1. Extraiga la ACF y PACF de la serie del Reto y defina la ventana $L$ justificada.
2. Identifique las periodicidades dominantes con el periodograma.
3. Si su Reto cuenta con variables externas, determine si actúan como indicadores líderes usando CCF.

In [ ]:
# ==============================================================================
# TODO 1: ACF y PACF sobre la serie del Reto
# ==============================================================================
# Grafique la ACF y PACF e identifique los rezagos significativos que exceden las bandas.


In [ ]:
# ==============================================================================
# TODO 2: Periodograma y ciclos dominantes
# ==============================================================================
# Calcule el periodograma y reporte los picos de frecuencia y periodos asociados.


## 5. Conclusiones y Pregunta Abierta
- **Conclusión:** La PACF revela qué rezagos pasados retienen información independiente no redundante, y el periodograma determina con exactitud matemática el periodo estacional $s$.
- **Pregunta para Semana 07:** ¿Cómo formulamos un modelo econométrico SARIMAX que use estos parámetros específicos ($p, d, q, s$) para generar intervalos de pronóstico al 95%?
